In [ ]:
# Load dataset
import pandas as pd
df = pd.read_csv('heart.csv')
df.head()

In [ ]:
# Manually encode text columns into numbers
df.Sex = df.Sex.replace({"M": 1, "F": 2})
df.RestingECG = df.RestingECG.replace({"Normal": 0, "ST": 1, "LVH": 2})
df.ExerciseAngina = df.ExerciseAngina.replace({"N": 0, "Y": 1})
df.ST_Slope = df.ST_Slope.replace({"Up": 0, "Flat": 1, "Down": 2})


In [ ]:
# One-hot encode the multi-category ChestPainType column
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first')
ChestPainType_encoded = ohe.fit_transform(df[['ChestPainType']]).toarray()

In [ ]:
# Separate the target column (HeartDisease) from features
y = df['HeartDisease']
df = df.drop(columns=['ChestPainType', 'HeartDisease'])

In [ ]:
# Add the newly encoded columns back into the dataframe
df = pd.concat([df, pd.DataFrame(ChestPainType_encoded, columns=ohe.get_feature_names_out(), index=df.index)], axis=1)

In [ ]:
# Split dataset into 80% training and 20% testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2)


In [ ]:
# Scale features so they all have the same range
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train a standalone Decision Tree and check its accuracy
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(X_scaled, y_train)
print("Decision Tree accuracy:", dt.score(X_test_scaled, y_test))

Decision Tree accuracy: 0.782608695652174


In [ ]:
# Train a Bagging model using Decision Trees and check its accuracy
from sklearn.ensemble import BaggingClassifier
bag_model = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, max_samples=0.8, oob_score=True, random_state=0)
bag_model.fit(X_scaled, y_train)
print("Bagging Test accuracy:", bag_model.score(X_test_scaled, y_test))

Bagging Test accuracy: 0.8695652173913043
